[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/csg_raytracer/csg_raytracer.ipynb)

# Solids Cut from Quadrics, Ray Traced

A ruled hyperboloid cut off by two planes, a bowl hollowed out and cut open, and a mirror sphere stand on a checkered floor, lit by a lamp that casts their shadows, while a camera circles them. Every solid is the intersection of a few quadric solids, and the floor and the planes that cut are quadrics too, so one ray tracer shows all of them.

A solid quadric is a map from points to planes, `Plane <- Point`, whose pairing with a point, `quadric(point) & point`, is zero on its surface and negative inside, and at a point of the surface the plane it gives is the tangent plane. A half-space is the product of its plane's pairing with the plane at infinity's, `plane * (mv.w & Point)` symmetrized; negating a quadric swaps inside and outside, which is how the bowl loses its inside. Along a ray the pairing is a quadratic in the distance, and the surface a ray sees first is a root of one solid's quadric that lies inside all its others. The tracer runs on the GPU, in a shader whose expressions are numga's.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.quadrics.csg_raytracer import render
from examples.util.wgsl import Shader, image_figure, supersampled

In [ ]:
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
# Points and directions are antivectors, planes are vectors, and a direction is a point at infinity.
Point = ga.gatype.antivector()                                                 # Antivector-
Plane = ga.gatype.vector()                                                     # Vector+
Motor = ga.gatype.rotor()                                                      # Rotor+
Quadric = ga.gatype((Plane, Point))
origin = mv.point([0, 0, 0])                                                   # [] Point-
# The planes through the origin square to each axis.
axes = stack((mv.x, mv.y, mv.z))                                               # [3] Plane+

## 1. A ruled hyperboloid

A hyperboloid of one sheet is an ellipsoid with the pairing along its axis subtracted, and through every point of its waist pass two straight lines lying in its surface. Turning the waist point and both directions about z together gives them all.

In [ ]:
def ellipsoid(semi_axes: np.ndarray) -> Quadric:
    """A solid ellipsoid about the origin: each axis's squared pairing over its semi-axis squared, less the plane at
    infinity's."""
    return (axes * (axes & Point) / semi_axes**2).sum(axis=-1) - mv.w * (mv.w & Point)


def hyperboloid(semi_axes: np.ndarray) -> Quadric:
    """A solid ruled hyperboloid of one sheet about the origin, its waist around z: as an ellipsoid, with the pairing
    along z subtracted."""
    signs = np.array([1, 1, -1])
    return (axes * (axes & Point) * signs / semi_axes**2).sum(axis=-1) - mv.w * (mv.w & Point)


def ruling_lines(semi_axes: np.ndarray, n_rulings: int) -> tuple[Point, Point, Point]:
    """Points around the waist of a circular hyperboloid, and the directions of the two straight lines through each
    that lie in its surface: the point and its directions turned about z together."""
    radius, _, height = semi_axes
    turns = (mv.xy * (-np.linspace(0, 2 * np.pi, n_rulings, endpoint=False) / 2)).exp()   # [rulings] Motor+
    waist = turns >> mv.point([radius, 0, 0])                                             # [rulings] Point-
    rising = turns >> mv.direction([0, radius, height])                                   # [rulings] Point-
    falling = turns >> mv.direction([0, radius, -height])                                 # [rulings] Point-
    return waist, rising, falling


# The hyperboloid's semi-axes, the rulings through its waist, and the offsets along them.
waist_axes = np.array([0.55, 0.55, 0.85])
n_rulings = 12
offsets = np.linspace(-2, 2, 5)[:, None]

waist, rising, falling = ruling_lines(waist_axes, n_rulings)                   # [rulings] Point-, Point-, Point-
# Points along both families of lines, and the hyperboloid's pairing with them: zero, for they lie in its surface.
on_rulings = waist + stack([rising, falling])[:, None] * offsets               # [families, offsets, rulings] Point-
along_rulings = hyperboloid(waist_axes)(on_rulings) & on_rulings               # [families, offsets, rulings] Scalar+

## 2. Solids as intersections

Each solid is a list of quadrics, all placed by one motor, `motor >> quadric(motor << Point)`: each point moved back, measured, and its plane moved forward. The shader reads every solid's range of quadrics, its colour, how much it mirrors, and two planes whose unit steps lay a checker over it.

In [ ]:
def half_space(plane: Plane) -> Quadric:
    """The points on a plane's negative side, as the quadric that pairs them with the plane and with the plane at
    infinity, symmetrized."""
    return (plane * (mv.w & Point) + mv.w * (plane & Point)) / 2


# Where each solid stands, as a direction from the origin, and how it is tilted; the bowl's radii, and the plane that
# cuts it open; the mirror's radius.
hyperboloid_place, hyperboloid_tilt = mv.direction([-1.2, 0.4, 0.85]), np.radians(12.0)
bowl_place, bowl_tilts = mv.direction([1.1, -0.2, 0.75]), np.radians([35.0, 15.0])
bowl_radius, bowl_inner_radius = 0.75, 0.6
bowl_cut = mv.y * 0.15 + mv.z * 0.98 - mv.w * 0.1
mirror_place, mirror_radius = mv.direction([0.0, 0.1, 0.55]), 0.55

# Each solid is placed by one motor: the translation along its place, `((mv.w ^ place.dual()) / 2).exp()`, after
# its tilts.
lift = ((mv.w ^ hyperboloid_place.dual()) / 2).exp() * (mv.xz * (-hyperboloid_tilt / 2)).exp()   # [] Motor+
bowl_shift = ((mv.w ^ bowl_place.dual()) / 2).exp()                                              # [] Motor+
bowl_tilt = (mv.yz * (-bowl_tilts[0] / 2)).exp() * (mv.xz * (-bowl_tilts[1] / 2)).exp()          # [] Motor+
turn_bowl = bowl_shift * bowl_tilt                                                               # [] Motor+
mirror_shift = ((mv.w ^ mirror_place.dual()) / 2).exp()                                          # [] Motor+
height = waist_axes[2]
# The hyperboloid between the planes a half-height above and below its waist.
hyperboloid_parts = stack([hyperboloid(waist_axes), half_space(mv.z - mv.w * height),
                           half_space(-mv.z - mv.w * height)])                                   # [parts] Plane+ <- Point-
# The bowl: inside the outer ellipsoid, outside the inner one, below the cut.
bowl_parts = stack([ellipsoid(np.full(3, bowl_radius)), -ellipsoid(np.full(3, bowl_inner_radius)),
                    half_space(bowl_cut.normalized())])                                          # [parts] Plane+ <- Point-
mirror_parts = ellipsoid(np.full(3, mirror_radius)).reshape(1)                                   # [parts] Plane+ <- Point-
# The floor: everything below the plane z = 0.
floor_parts = half_space(mv.z).cast(Quadric).reshape(1)                                          # [parts] Plane+ <- Point-
# Each solid's parts placed by its motor: every point moved back, measured, and its plane moved forward.
shapes = [lift >> hyperboloid_parts(lift << Point), turn_bowl >> bowl_parts(turn_bowl << Point),
          mirror_shift >> mirror_parts(mirror_shift << Point), floor_parts]
counts = np.array([shape.shape[0] for shape in shapes])
ends = np.cumsum(counts)
quadrics = concatenate(shapes)                                                                   # [quadrics] Plane+ <- Point-
# Stripes crossing the hyperboloid's rulings, and tiles on the floor.
stripes, tiles = 8.0, 1.4
solids = dict(first=ends - counts, end=ends,
              colour=np.array([[0.84, 0.48, 0.24], [0.15, 0.45, 0.55], [0.95, 0.95, 0.96], [0.88, 0.87, 0.85]]),
              reflectivity=np.array([0.0, 0.0, 0.88, 0.0]),
              across=stack([lift >> (mv.x + mv.z) * stripes, 0 * mv.x, 0 * mv.x, mv.x * tiles]),
              along=stack([lift >> (mv.y - mv.z) * stripes, 0 * mv.y, 0 * mv.y, mv.y * tiles]))

## 3. The shader

Each invocation follows one camera ray. Along it a quadric is a quadratic in the distance $t$, $a t^2 + 2 b t + c$, its coefficients the quadric's pairings of the ray's origin and direction; a half-space has no square term, and the stable form of the roots gives its one root without a case of its own. The tangent plane lights the surface by its pairing with the lamp's direction, `normal & lamp`, and reflects the view ray, `normal >> direction`, for the glint and for the mirror's second look.

In [ ]:
source = supersampled() + """
// A solid: the intersection of its quadrics, `quadrics[first]` up to `quadrics[end]`, each negative
// inside it; its colour, the share of light it reflects as a mirror, and two planes whose unit steps
// lay a checker over it.
struct Solid {
    first: u32,
    end: u32,
    colour: vec3<f32>,
    reflectivity: f32,
    across: Plane,
    along: Plane,
}

var<storage, read> solids: array<Solid>;
var<storage, read> quadrics: array<Quadric>;
var<uniform> camera: Motor;
var<uniform> eye: Point;
var<uniform> lamp: Point;
var<uniform> half_view: f32;
alias Sample = vec3<f32>;

// Distances beyond the scene, and the step a ray takes off a surface before it looks again.
const far = 1e6;
const clearance = 1e-3;
// How far a point may lie outside a quadric and still count as inside, for points on two surfaces.
const tolerance = 1e-4;
// The look: light where the lamp does not reach, a shadow's light, the checker's darkening, and the glint.
const ambient = 0.28;
const shadow = 0.3;
const checker = 0.15;
const shininess = 40.0;
const gloss = 0.3;
const horizon = vec3<f32>(0.80, 0.84, 0.90);
const zenith = vec3<f32>(0.38, 0.52, 0.72);

struct Hit {
    distance: f32,
    tangent: Plane,
    solid: u32,
}

// Whether a point lies inside all of a solid's quadrics but one.
fn inside_others(solid: Solid, own: u32, point: Point) -> bool {
    for (var index = solid.first; index < solid.end; index++) {
        if (index != own && (quadrics[index](point) & point) > tolerance) {
            return false;
        }
    }
    return true;
}

// Where a ray first meets a solid's surface: on one of its quadrics, and inside all the others.
fn first_hit(origin: Point, direction: Point) -> Hit {
    var hit = Hit(far, mv.z, 0u);
    for (var index = 0u; index < arrayLength(&solids); index++) {
        let solid = solids[index];
        for (var own = solid.first; own < solid.end; own++) {
            let quadric = quadrics[own];
            // Along the ray the quadric is a quadratic in the distance: its square, twice its linear and its
            // constant coefficient are a, b and c.
            let a = quadric(direction) & direction;
            let b = ((quadric(direction) & origin) + (quadric(origin) & direction)) / 2;
            let c = quadric(origin) & origin;
            let discriminant = b * b - a * c;
            if (discriminant < 0) {
                continue;
            }
            // Both roots without cancellation; a half-space, with no square term, has only the second.
            let q = -(b + select(-1.0, 1.0, b >= 0) * sqrt(discriminant));
            let roots = vec2<f32>(select(q / a, -1.0, a == 0), c / q);
            for (var which = 0u; which < 2u; which++) {
                let distance = roots[which];
                let point = origin + direction * distance;
                if (distance > clearance && distance < hit.distance && inside_others(solid, own, point)) {
                    // The quadric's plane at a point of its surface is the tangent plane there.
                    hit = Hit(distance, quadric(point), index);
                }
            }
        }
    }
    return hit;
}

struct Sight {
    colour: vec3<f32>,
    point: Point,
    normal: Plane,
    reflectivity: f32,
}

// What a ray sees: the solid it meets, lit by the lamp, shadowed, checkered and glinting; or the sky.
fn sight(origin: Point, direction: Point) -> Sight {
    let hit = first_hit(origin, direction);
    if (hit.distance >= far) {
        return Sight(mix(horizon, zenith, clamp(mv.z & direction, 0.0, 1.0)), origin, mv.z, 0.0);
    }
    let solid = solids[hit.solid];
    let point = origin + direction * hit.distance;
    let normal = hit.tangent / (hit.tangent | hit.tangent).square_root();
    // The lamp's light falls by the cosine of its direction off the normal, a plane paired with a direction.
    let lit = max(normal & lamp, 0.0) * select(1.0, shadow, first_hit(point + lamp * clearance, lamp).distance < far);
    let parity = abs((floor(solid.across & point) + floor(solid.along & point)) % 2.0);
    // The view ray reflected in the tangent plane, against the lamp: `-d.dual()` is the plane square to d.
    let glint = pow(max(-((normal >> direction).dual() & lamp), 0.0), shininess);
    let colour = solid.colour * (1 - checker * parity) * (ambient + (1 - ambient) * lit) + gloss * glint * lit;
    return Sight(colour, point, normal, solid.reflectivity);
}

fn unit(direction: Point) -> Point {
    return direction / (-(direction.dual() & direction)).square_root();
}

// What the ray through a place on the image sees: along the camera's own -z, turned by its motor.
fn sample(place: vec2<f32>) -> Sample {
    let across = screen(place) * half_view;
    let direction = unit(camera >> (-mv.z.dual() + mv.x.dual() * across.x + mv.y.dual() * across.y));
    let seen = sight(eye, direction);
    // A mirror shows what its reflection sees: the view ray reflected in the tangent plane.
    let reflected = seen.normal >> direction;
    let mirrored = sight(seen.point + reflected * clearance, reflected);
    return mix(seen.colour, mirrored.colour, seen.reflectivity);
}
"""

## 4. The camera's view

A camera is a motor: one at the origin looking along its own -z, backed off, tilted down, turned about the vertical, and carried to the target. The image comes from `supersampled()`, a turned grid of two samples a side in each pixel, averaged; the shader says what one sample sees, `fn sample(place: vec2<f32>) -> Sample`.

In [ ]:
target = mv.point([0.0, 0.2, 0.7])
distance, elevation, half_view = 4.6, np.radians(28.0), 0.42
lamp = mv.direction([0.5, -0.5, 0.7])
# The still's pixels, and the animation's.
image_shape, animation_shape = (450, 720), (330, 528)


def orbit(target: Point, distance: float, elevation: float, azimuth: np.ndarray) -> Motor:
    """A camera's motor, from one at the origin looking along its own -z with y up: backed off by `distance`, tilted
    down to look from `elevation` above the horizon, turned to `azimuth` about the vertical, and carried to the
    target."""
    back_off = (mv.zw * (distance / 2)).exp()                                  # [] Motor+
    pitch = (mv.yz * (-(np.pi / 2 - elevation) / 2)).exp()                    # [] Motor+
    turn = (mv.xy * (-azimuth / 2)).exp()                                      # [...] Motor+
    # The translation along the direction from the origin to the target, a point at infinity.
    carry = ((mv.w ^ (target - origin).dual()) / 2).exp()                     # [] Motor+
    return carry * turn * pitch * back_off


tracer = Shader(source, ga, dict(Point=Point, Plane=Plane, Motor=Motor, Quadric=Quadric), {})
still, = render.frames(tracer, quadrics, solids, orbit(target, distance, elevation, np.zeros(1)), lamp, half_view, image_shape)
save_figure(image_figure(still), "csg_raytracer");

## 5. Around the scene

The camera circles the target, each frame a fresh trace: the mirror shows the other solids as they come round, and the bowl's cut shows its hollow.

In [ ]:
n_frames, duration_ms = 96, 40
cameras = orbit(target, distance, elevation, np.linspace(0, 2 * np.pi, n_frames, endpoint=False))   # [frames] Motor+
show_animation(render.frames(tracer, quadrics, solids, cameras, lamp, half_view, animation_shape), "csg_raytracer", duration_ms)

In [ ]:
# checks
# Both families of straight lines through the waist lie in the hyperboloid's surface, and every camera looks at the target.
np.testing.assert_allclose(along_rulings.kernel, 0, atol=1e-12)
np.testing.assert_allclose(((cameras >> origin) + (cameras >> -mv.z.dual()) * distance - target).kernel, 0, atol=1e-12)